# Streaming remote CNDB trajectories with CNDBTools

CNDBTools can now stream indexed CNDB/HDF5 trajectories over HTTP using an internal OpenMiChroM backend. The full remote CNDB file is not downloaded. Instead, the backend uses HTTP Range requests to read embedded HDF5 index metadata and then fetch only the coordinate bytes needed for selected trajectories, frames, and bead ranges.

The embedded index is read once and can be cached locally. For contiguous, uncompressed frame datasets, coordinate reads fetch exact byte ranges for the requested bead interval.

Full mode uses a real ENCODE CNDB file of about 139 GB and requires internet access. Fast validation mode creates a small temporary CNDB and deterministic local HTTP range server, so default continuous integration remains offline.

## Reproducible execution modes

This notebook runs the scientifically complete workflow by default. Set the environment variable `OPENMICHROM_TUTORIAL_MODE=fast` for the reduced CPU validation used by `python scripts/validate.py complete`. Fast mode exercises the same setup, force construction, integration, analysis, and output paths with fewer steps; it is a smoke test, not a production simulation. Set `OPENMICHROM_TUTORIAL_PLATFORM` to override the default `CPU` platform.


In [ ]:
import os
from pathlib import Path

import numpy as np

TUTORIAL_FAST = os.environ.get("OPENMICHROM_TUTORIAL_MODE", "full").lower() == "fast"
TUTORIAL_PLATFORM = os.environ.get("OPENMICHROM_TUTORIAL_PLATFORM", "CPU")
np.random.seed(2026)
print(f"OpenMiChroM tutorial mode: {'fast smoke test' if TUTORIAL_FAST else 'full scientific run'}")
print(f"OpenMM platform: {TUTORIAL_PLATFORM}")


## Installation

For normal users after release:

```bash
pip install OpenMiChroM
```

For development from local repositories:

```bash
pip install -e .
```

The installation commands are shown for reference and are not executed by this notebook.

In [ ]:
import tempfile
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import h5py
import matplotlib.pyplot as plt
import numpy as np

from OpenMiChroM.CndbTools import CndbTools
from OpenMiChroM._cndb_stream.index import build_index


## Define the remote ENCODE CNDB file

This URL points to a large ENCODE CNDB/HDF5 file. The notebook opens it remotely and reads only the embedded index plus small coordinate ranges. It does not download the full file.

In [ ]:
ENCODE_URL = "https://encode-public.s3.amazonaws.com/2023/02/02/7f75d816-342a-4b49-adbd-aaa499dc5201/ENCFF161DID.cndb"
TRAJECTORY = "replica1_chr1"
CACHE_PATH = Path(".openmichrom-cache/ENCFF161DID.embedded-index.json.gz")
server = None
temporary_fixture = None

if TUTORIAL_FAST:
    temporary_fixture = tempfile.TemporaryDirectory(prefix="openmichrom-stream-tutorial-")
    fixture_dir = Path(temporary_fixture.name)
    fixture_path = fixture_dir / "tutorial.cndb"
    index_path = fixture_dir / "tutorial.index.json"
    with h5py.File(fixture_path, "w") as handle:
        handle.attrs["format"] = "cndb"
        handle.attrs["format_version"] = "1.0.0"
        handle["types"] = np.array([b"A1", b"B1"] * 50)
        for frame in range(1, 5):
            handle[str(frame)] = np.arange(300, dtype=np.float32).reshape(100, 3) + frame
    build_index(fixture_path, index_path)
    payload = fixture_path.read_bytes()

    class RangeHandler(BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.0"
        def log_message(self, format, *args):
            return
        def do_GET(self):
            start_text, stop_text = self.headers["Range"][6:].split("-", 1)
            start, stop = int(start_text), int(stop_text)
            selected = payload[start:stop + 1]
            self.send_response(206)
            self.send_header("Content-Range", f"bytes {start}-{stop}/{len(payload)}")
            self.send_header("Content-Length", str(len(selected)))
            self.end_headers()
            self.wfile.write(selected)

    server = ThreadingHTTPServer(("127.0.0.1", 0), RangeHandler)
    server.daemon_threads = True
    threading.Thread(target=server.serve_forever, daemon=True).start()
    ENCODE_URL = f"http://127.0.0.1:{server.server_port}/tutorial.cndb"
    TRAJECTORY = None


## Open the remote CNDB with CNDBTools

The first run may take longer because the embedded index is read from the remote file. Later runs can reuse `CACHE_PATH`. The cache stores index metadata only, not the 139 GB coordinate file.

In [ ]:
if TUTORIAL_FAST:
    tools = CndbTools().load(ENCODE_URL, index_path=index_path)
else:
    tools = CndbTools.from_remote(
        h5_url=ENCODE_URL,
        trajectory=TRAJECTORY,
        index_cache_path=CACHE_PATH,
    )

print("Current trajectory:", tools.current_trajectory)
print("Frames:", tools.Nframes)
print("Beads:", tools.Nbeads)
print("Available trajectories:", len(tools.trajectories))
print("Format:", tools.format_name, tools.format_version, tools.format_status)
print("Stream stats:", tools.stream_stats())
assert tools.Nframes > 0 and tools.Nbeads > 0


## Check cache behavior

If `index_cache_hit` is `False`, this session read the embedded index from the remote HDF5 file and wrote the local cache. If it is `True`, CNDBTools reused the local cache and avoided re-reading the embedded index payload.

In [ ]:
print("Index cache hit:", tools.stream_stats().get("index_cache_hit"))


## Read one small bead range

This reads frame 1 and beads 0:10. The expected array shape is `(1, 10, 3)`.

Expected coordinate payload: `10 beads * 3 coordinates * 4 bytes/float32 = 120 bytes`.

In [ ]:
before = tools.stream_stats()["data_bytes_read"]
coords = tools.xyz(frames=[1], beadSelection=range(0, 10))
after = tools.stream_stats()["data_bytes_read"]
expected = 10 * 3 * np.dtype("float32").itemsize
print("coords shape:", coords.shape)
print("coordinate bytes for this read:", after - before)
print("expected coordinate bytes:", expected)
assert coords.shape == (1, 10, 3)
assert after - before == expected


## Read several frames

Now read beads 0:100 from four selected frames. The expected array shape is `(4, 100, 3)`.

Expected coordinate payload: `4 frames * 100 beads * 3 coordinates * 4 bytes = 4,800 bytes`.

In [ ]:
frames = [1, 2, 3, 4] if TUTORIAL_FAST else [1, 10, 100, 1000]
beads = range(0, min(100, tools.Nbeads))
before = tools.stream_stats()["data_bytes_read"]
xyz = tools.xyz(frames=frames, beadSelection=beads)
after = tools.stream_stats()["data_bytes_read"]
expected_bytes = len(frames) * len(beads) * 3 * np.dtype("float32").itemsize
print("xyz shape:", xyz.shape)
print("coordinate bytes for this read:", after - before)
print("expected coordinate bytes:", expected_bytes)
assert xyz.shape == (len(frames), len(beads), 3)
assert after - before == expected_bytes


## Simple analysis: radius of gyration

The returned `xyz` array can be used with normal NumPy analysis or OpenMiChroM analysis helpers. Here we compute a small radius of gyration series for the selected bead range.

In [ ]:
rg_values = tools.compute_RG(xyz)
for frame, rg in zip(frames, rg_values):
    print(frame, rg)
assert np.all(np.isfinite(rg_values))


## Plot radius of gyration versus frame

In [ ]:
plt.figure()
plt.plot(frames, rg_values, marker="o")
plt.xlabel("Frame")
plt.ylabel("Radius of gyration")
plt.title(f"{TRAJECTORY or 'local fixture'}: beads 0-{len(beads)}")
plt.show()


## Plot a 3D structure subset

This plot uses only the first 100 beads from one frame, keeping the visualization lightweight.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
frame0 = xyz[0]
ax.plot(frame0[:, 0], frame0[:, 1], frame0[:, 2], marker="o", linewidth=1)
ax.set_title(f"frame {frames[0]}, beads 0-{len(beads)}")
plt.show()


## Non-contiguous bead selections

Contiguous ranges are most efficient. For best performance, use `range(start, stop)` or `slice(start, stop)` when possible.

Non-contiguous selections may read the smallest enclosing bead range and then subset in memory. For example, selecting beads `[0, 10, 20, 30]` may read beads 0:31 before returning only the requested four beads.

In [ ]:
selected = [0, 10, 20, 30]
before = tools.stream_stats()["data_bytes_read"]
xyz_selected = tools.xyz(frames=[1], beadSelection=selected)
after = tools.stream_stats()["data_bytes_read"]
print("selected shape:", xyz_selected.shape)
print("coordinate bytes for this read:", after - before)
assert xyz_selected.shape == (1, len(selected), 3)


## What was downloaded?

- The full 139 GB CNDB file was not downloaded.
- The embedded index may be read once. For this ENCODE file it is about 25.5 MB.
- With `index_cache_path`, later runs can show zero remote index bytes because the parsed index is loaded locally.
- Coordinate data bytes should match the requested subset for contiguous bead ranges.
- `tools.stream_stats()` is the best way to verify what happened in the current session.

## Troubleshooting

**Import error**

Remote streaming is included with OpenMiChroM. If imports fail, confirm that OpenMiChroM is installed in the active Python environment:

```bash
pip install OpenMiChroM
```

**Slow first open**

This is expected for large indexed files. The embedded index is being read. Use `index_cache_path` so later opens can reuse the cache.

**Server does not support Range requests**

The backend should fail rather than accidentally download the full file if the server ignores HTTP Range requests.

**Chunked or compressed frames**

Direct coordinate byte reads currently require contiguous, uncompressed frame datasets. Chunked/compressed direct reads are planned for future work.

In [ ]:
tools.close()
if server is not None:
    server.shutdown()
    server.server_close()
if temporary_fixture is not None:
    temporary_fixture.cleanup()
print('CNDB resources closed')
